# Modelo Medallion


Capas:
1. **Bronze** – ingesta cruda, tipos todos `string`.
2. **Silver** – limpieza: encoding, validación de coordenadas, deduplicación, columnas derivadas.
3. **Gold** – tablas agregadas listas para consulta / dashboard.


## 1. Instalación y sesión de Spark

In [ ]:
!pip install -q pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
import pandas as pd
datos = pd.read_csv('/content/data.csv')
print(datos.shape)
datos.head(30)

(472801, 11)


,departamento,provincia,distrito,latitud,longitud,fecha_hora,descripcion,categoria_inei,clasificacion_delito,gravedad_real,gravedad_reportada
0,CAJAMARCA,JAEN,JAEN,-5.769360,-78.785544,2025-11-25 15:59:27,Presencié cómo una persona me golpeó y robó mi...,Contra el patrimonio,NaN,alta,NaN
1,SAN MARTIN,TOCACHE,UCHIZA,-8.475570,-76.438240,NaN,Presencié cómo una persona me golpeó y robó mi...,Contra el patrimonio,Robo agravado,alta,alta
2,SAN MARTIN,RIOJA,SAN FERNANDO,NaN,-77.255923,NaN,Un sujeto me golpeó y robó mis pertenencias en...,Contra el patrimonio,Robo agravado,alta,alta
3,LIMA,NaN,SAN LUIS,-12.061834,-77.001525,2025-08-06 01:28:29,Presencié cómo una persona me hizo transferir ...,Contra el patrimonio,Estafa,baja,baja
4,LIMA,CAÑETE,COAYLLO,NaN,NaN,2025-05-26 01:04:39,Presencié cómo una persona se llevó mi bicicle...,Contra el patrimonio,NaN,media,alta
5,LIMA,LIMA,SANTIAGO DE SURCO,NaN,-76.961296,2025-12-18 19:15:04,Se reporta que un individuo me quitó el celula...,Contra el patrimonio,Robo,alta,NaN
6,LIMA,HUAROCHIRI,RICARDO PALMA,-11.914828,-76.614063,2025-09-03 18:02:47,NaN,Contra la seguridad pública,Terrorismo,alta,alta
7,LIMA,LIMA,VILLA EL SALVADOR,-12.229749,-76.961103,2025-05-19 16:07:41,Presencié cómo una persona se llevó mi bicicle...,Contra el patrimonio,Hurto,media,media
8,AREQUIPA,AREQUIPA,SAN JUAN DE SIGUAS,-16.390166,-72.152208,2025-07-25 04:51:00,Un sujeto dejó un paquete sospechoso cerca de ...,Contra la seguridad pública,Terrorismo,alta,alta
9,NaN,HUANCAYO,VIQUES,-12.159125,NaN,2025-06-04 04:53:48,Un sujeto vendía sustancias ilícitas en la esq...,Contra la seguridad pública,NaN,media,media


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T

spark = (
    SparkSession.builder
    .appName("BDA-Medallion-Delitos")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark

## 2. Carga del archivo

In [ ]:
RAW_PATH = "/content/data.csv"

print("Archivo cargado:", RAW_PATH)

Archivo cargado: /content/data.csv


In [ ]:
LAKEHOUSE    = "/content/lakehouse"
BRONZE_PATH  = f"{LAKEHOUSE}/bronze/delitos"
SILVER_PATH  = f"{LAKEHOUSE}/silver/delitos"
GOLD_PATH    = f"{LAKEHOUSE}/gold"

## 3. BRONZE — ingesta cruda


In [ ]:
bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .csv(RAW_PATH)
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source_file", F.lit(RAW_PATH))
)

print("Total de filas:", bronze_df.count())
bronze_df.printSchema()
bronze_df.show(5, truncate=60)

bronze_df.write.mode("overwrite").parquet(BRONZE_PATH)

Total de filas: 4800000
root
 |-- departamento: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- latitud: string (nullable = true)
 |-- longitud: string (nullable = true)
 |-- fecha_hora: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- categoria_inei: string (nullable = true)
 |-- clasificacion_delito: string (nullable = true)
 |-- gravedad_real: string (nullable = true)
 |-- gravedad_reportada: string (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)

+------------+---------+--------------+----------+----------+-------------------+------------------------------------------------------------+--------------------+--------------------+-------------+------------------+--------------------------+-----------------+
|departamento|provincia|      distrito|   latitud|  longitud|         fecha_hora|                                                 d

## 4. SILVER — limpieza y tipado
- Repara el mojibake en columnas de texto.
- Castea tipos: `double` (coordenadas), `timestamp` (fecha), `boolean` (flags de ruido).
- Descompone `ruta_imagen` / `ruta_imagen_real` en `fuente` (dataset abierto vs. difusión sintética) y `categoria`.
- Deriva columnas de fecha (año, mes, día, hora, día de semana) para facilitar el Gold.
- Elimina duplicados exactos y filas sin fecha/descripción.

In [ ]:
def fix_mojibake(colname):

  return F.when(
      F.col(colname).rlike("Ã"),
      F.decode(F.encode(F.col(colname), "ISO-8859-1"), "UTF-8"),
  ).otherwise(F.col(colname))


text_cols = [
    "departamento",
    "provincia",
    "distrito",
    "descripcion",
    "categoria_inei",
    "clasificacion_delito",
]

silver_df = bronze_df
for c in text_cols:
  silver_df = silver_df.withColumn(c, F.trim(fix_mojibake(c)))

silver_df = (
    silver_df

    .withColumn(
        "fecha_hora_limpia",
        F.when(
            (F.col("fecha_hora").isNull())
            | (F.lower(F.trim(F.col("fecha_hora"))) == "nan")
            | (F.length(F.trim(F.col("fecha_hora"))) == 0),
            None,
        ).otherwise(F.trim(F.col("fecha_hora"))),
    )

    .withColumn("latitud", F.col("latitud").cast("double"))
    .withColumn("longitud", F.col("longitud").cast("double"))
    .withColumn(
        "fecha_hora",
        F.to_timestamp(F.col("fecha_hora_limpia"), "yyyy-MM-dd HH:mm:ss"),
    )
    .drop("fecha_hora_limpia")
    .withColumn("departamento", F.upper("departamento"))
    .withColumn("provincia", F.upper("provincia"))
    .withColumn("distrito", F.upper("distrito"))
    .withColumn("gravedad_real", F.lower("gravedad_real"))
    .withColumn("gravedad_reportada", F.lower("gravedad_reportada"))

    .na.fill(
        {
            "departamento": "DESCONOCIDO",
            "provincia": "DESCONOCIDO",
            "distrito": "DESCONOCIDO",
            "descripcion": "Sin descripción",
            "categoria_inei": "Otros",
            "clasificacion_delito": "No especificado",
            "gravedad_real": "media",
            "gravedad_reportada": "media",
        }
    )
    .na.fill({"latitud": 0.0, "longitud": 0.0})

    .withColumn("anio", F.year("fecha_hora"))
    .withColumn("mes", F.month("fecha_hora"))
    .withColumn("dia", F.dayofmonth("fecha_hora"))
    .withColumn("hora", F.hour("fecha_hora"))
    .withColumn("dia_semana", F.date_format("fecha_hora", "E"))
)

coord_ok = F.col("latitud").between(-18.5, 0.5) & F.col("longitud").between(
    -81.5, -68.5
)
silver_df = silver_df.withColumn("coordenadas_validas", coord_ok)

silver_df = (
    silver_df.filter(F.col("fecha_hora").isNotNull())
    .filter(
        F.col("descripcion").isNotNull()
        & (F.length("descripcion") > 0)
        & (F.col("descripcion") != "Sin descripción")
    )
    .dropDuplicates(
        ["departamento", "provincia", "distrito", "fecha_hora", "descripcion"]
    )
)

print("CONTEO DE NULOS DESPUÉS DE LA LIMPIEZA")
for c in silver_df.columns:
  col_type = silver_df.schema[c].dataType
  if isinstance(col_type, (T.DoubleType, T.FloatType)):
    n_nulls = silver_df.filter(
        F.col(c).isNull() | F.isnan(F.col(c))
    ).count()
  else:
    n_nulls = silver_df.filter(F.col(c).isNull()).count()

  print(f"{c}: {n_nulls} nulos")

print("\nFilas Silver finales:", silver_df.count())
print(
    "Coordenadas inválidas:",
    silver_df.filter(~F.col("coordenadas_validas")).count(),
)

silver_df.write.mode("overwrite").partitionBy("anio", "mes").parquet(SILVER_PATH)

--- CONTEO DE VALORES NULOS POR COLUMNA ---
departamento: 0 nulos
provincia: 0 nulos
distrito: 0 nulos
latitud: 0 nulos
longitud: 0 nulos
fecha_hora: 0 nulos
descripcion: 0 nulos
categoria_inei: 0 nulos
clasificacion_delito: 0 nulos
gravedad_real: 0 nulos
gravedad_reportada: 0 nulos
_ingestion_ts: 0 nulos
_source_file: 0 nulos
anio: 0 nulos
mes: 0 nulos
dia: 0 nulos
hora: 0 nulos
dia_semana: 0 nulos
coordenadas_validas: 0 nulos

Filas Silver finales: 3013738
Coordenadas inválidas: 452311


## 5. GOLD — tablas de negocio
Cuatro tablas:
1. `gold_incidencia` — casos por ubicación, tiempo, categoría y gravedad.
2. `gold_confiabilidad` — qué tan seguido coincide la gravedad reportada por el ciudadano con la real.
3. `gold_calidad` — ruido de texto/imagen por fuente (dataset abierto vs. difusión sintética).
4. `gold_serie_temporal` — serie diaria de casos por categoría INEI.

In [ ]:
silver_df.createOrReplaceTempView("silver_delitos")

gold_incidencia = spark.sql("""
    SELECT departamento, provincia, distrito, anio, mes,
           categoria_inei, clasificacion_delito, gravedad_real,
           COUNT(*) AS total_casos
    FROM silver_delitos
    WHERE coordenadas_validas = true
    GROUP BY departamento, provincia, distrito, anio, mes,
             categoria_inei, clasificacion_delito, gravedad_real
""")

gold_confiabilidad = spark.sql("""
    SELECT clasificacion_delito, gravedad_real, gravedad_reportada,
           COUNT(*) AS total_casos,
           ROUND(100 * SUM(CASE WHEN gravedad_real = gravedad_reportada THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_coincidencia
    FROM silver_delitos
    GROUP BY clasificacion_delito, gravedad_real, gravedad_reportada
""")

gold_serie_temporal = spark.sql("""
    SELECT DATE(fecha_hora) AS fecha, categoria_inei, COUNT(*) AS total_casos
    FROM silver_delitos
    GROUP BY DATE(fecha_hora), categoria_inei
    ORDER BY fecha
""")

for name, df in [("incidencia", gold_incidencia), ("confiabilidad", gold_confiabilidad),
                  ("serie_temporal", gold_serie_temporal)]:
    df.write.mode("overwrite").parquet(f"{GOLD_PATH}/{name}")
    print(name, "->", df.count(), "filas")

incidencia -> 1208955 filas
confiabilidad -> 286 filas
serie_temporal -> 2190 filas


## 6. Query
Top distritos con más casos de gravedad **alta** en 2025.

In [ ]:
gold_incidencia.createOrReplaceTempView("gold_incidencia")

respuesta = spark.sql("""
    SELECT departamento, clasificacion_delito, SUM(total_casos) AS casos_2025
    FROM gold_incidencia
    WHERE anio = 2025 AND gravedad_real = 'alta'
    GROUP BY departamento, clasificacion_delito
    ORDER BY casos_2025 DESC
    LIMIT 15
""")
respuesta.show(truncate=False)

+------------+------------------------+----------+
|departamento|clasificacion_delito    |casos_2025|
+------------+------------------------+----------+
|LIMA        |Robo                    |81292     |
|LIMA        |Robo agravado           |80578     |
|LIMA        |No especificado         |38096     |
|DESCONOCIDO |Robo agravado           |36825     |
|DESCONOCIDO |Robo                    |36666     |
|LIMA        |Homicidio               |21383     |
|LIMA        |Tenencia ilegal de armas|19845     |
|LIMA        |Terrorismo              |19671     |
|LAMBAYEQUE  |Robo                    |18723     |
|LAMBAYEQUE  |Robo agravado           |18261     |
|DESCONOCIDO |No especificado         |17334     |
|LA LIBERTAD |Robo agravado           |15535     |
|LA LIBERTAD |Robo                    |15391     |
|AREQUIPA    |Robo agravado           |14810     |
|AREQUIPA    |Robo                    |14704     |
+------------+------------------------+----------+

